# CNN hyperparameter search

This notebook performs a lightweight random search for the CNN model.

The selection rule is:

```text
choose the configuration with the lowest validation MAE
```

The test set is not used during hyperparameter selection. It is evaluated only once at the end using the best validation configuration.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path
import sys
import json
import random

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

import keras
from keras import backend as K
from keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    SpatialDropout1D,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    Concatenate,
    Dense,
    Dropout,
)
from keras.models import Model
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "util.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing util.py and data/")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from util import get_train_test, RANDOM_SEED

OUTPUT_DIR = PROJECT_ROOT / "model" / "CNN" / "outputs" / "cnn_hyperparameter_search_30_5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)

I0000 00:00:1777829819.131211 1765399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777829819.131512 1765399 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1777829819.923866 1765399 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777829819.924071 1765399 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Project root: /home/hugo/Desktop/Neural-Networks-Forecasting
Output dir: /home/hugo/Desktop/Neural-Networks-Forecasting/model/CNN/outputs/cnn_hyperparameter_search_30_5


## Search configuration

The default search is centered on the validated `30 → 5` setup. Increase `N_TRIALS` for a more complete search or reduce it temporarily for a quick smoke test.

In [2]:
INPUT_WINDOW = 30
OUTPUT_WINDOW = 5
VALIDATION_RATIO = 0.10

N_TRIALS = 15
EPOCHS = 100

## Data preparation

As in the CNN Deep notebook, the scaler is fitted only on the training split and then applied to validation and test.

In [3]:
def split_train_val(X_train, y_train, val_ratio=0.10):
    val_size = int(len(X_train) * val_ratio)
    if val_size <= 0:
        raise ValueError("Validation split is empty. Increase training size or val_ratio.")

    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    return X_train_final, y_train_final, X_val, y_val


def scale_X_only(X_train, X_val, X_test):
    n_train, window, n_assets = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    scaler = StandardScaler()
    X_train_2d = X_train.reshape(n_train, -1)
    X_val_2d = X_val.reshape(n_val, -1)
    X_test_2d = X_test.reshape(n_test, -1)

    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, window, n_assets)
    X_val_scaled = scaler.transform(X_val_2d).reshape(n_val, window, n_assets)
    X_test_scaled = scaler.transform(X_test_2d).reshape(n_test, window, n_assets)
    return X_train_scaled, X_val_scaled, X_test_scaled


d = get_train_test(INPUT_WINDOW, OUTPUT_WINDOW)

X_train_raw, y_train_raw = d.X_train, d.y_train
X_test_raw, y_test = d.X_test, d.y_test

X_train_raw, y_train, X_val_raw, y_val = split_train_val(
    X_train_raw, y_train_raw, val_ratio=VALIDATION_RATIO
)
X_train, X_val, X_test = scale_X_only(X_train_raw, X_val_raw, X_test_raw)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

X_train: (13082, 30, 23)
y_train: (13082, 23)
X_val:   (1453, 30, 23)
y_val:   (1453, 23)
X_test:  (1616, 30, 23)
y_test:  (1616, 23)


## Hyperparameter space

The search varies filters, kernel sizes, dilation, dropout, dense layer sizes, learning rate and batch size. This is a controlled random search, not an exhaustive grid search.

In [4]:
def sample_config(trial_id):
    cfg = {
        "filters_1": random.choice([32, 64, 96]),
        "filters_2": random.choice([32, 64, 96]),
        "filters_3": random.choice([64, 96, 128]),
        "kernel_1": random.choice([3, 5]),
        "kernel_2": random.choice([3, 5, 7]),
        "kernel_3": random.choice([3, 5]),
        "dilation_2": random.choice([1, 2]),
        "dilation_3": random.choice([2, 4]),
        "spatial_dropout": random.choice([0.05, 0.10, 0.15, 0.20]),
        "dense_1": random.choice([64, 128, 256]),
        "dense_2": random.choice([32, 64, 128]),
        "dropout_1": random.choice([0.10, 0.20, 0.30]),
        "dropout_2": random.choice([0.05, 0.10, 0.20]),
        "learning_rate": random.choice([1e-3, 5e-4, 3e-4, 1e-4]),
        "batch_size": random.choice([64, 128, 256]),
    }
    cfg["trial_id"] = trial_id
    return cfg


def build_model(input_window, n_assets, cfg):
    inputs = Input(shape=(input_window, n_assets))

    x = Conv1D(cfg["filters_1"], cfg["kernel_1"], padding="causal", activation="relu")(inputs)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(cfg["spatial_dropout"])(x)

    x = Conv1D(
        cfg["filters_2"],
        cfg["kernel_2"],
        padding="causal",
        dilation_rate=cfg["dilation_2"],
        activation="relu",
    )(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(cfg["spatial_dropout"])(x)

    x = Conv1D(
        cfg["filters_3"],
        cfg["kernel_3"],
        padding="causal",
        dilation_rate=cfg["dilation_3"],
        activation="relu",
    )(x)
    x = BatchNormalization()(x)

    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    x = Concatenate()([avg_pool, max_pool])

    x = Dense(cfg["dense_1"], activation="relu")(x)
    x = Dropout(cfg["dropout_1"])(x)
    x = Dense(cfg["dense_2"], activation="relu")(x)
    x = Dropout(cfg["dropout_2"])(x)

    outputs = Dense(n_assets, activation="linear")(x)
    model = Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["learning_rate"]),
        loss="mae",
        metrics=["mae"],
    )
    return model

## Random search

Each trial is selected by validation MAE. During the search, partial results are saved to disk so the experiment can be inspected even if it is interrupted.

In [5]:
def train_trial(trial_id, cfg):
    K.clear_session()
    keras.utils.set_random_seed(RANDOM_SEED + trial_id)

    model = build_model(INPUT_WINDOW, X_train.shape[2], cfg)

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            min_delta=1e-6,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6,
        ),
    ]

    print("\n" + "=" * 80)
    print(f"Trial {trial_id}/{N_TRIALS}")
    print(json.dumps(cfg, indent=2))
    print("=" * 80)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=cfg["batch_size"],
        callbacks=callbacks,
        verbose=1,
        shuffle=True,
    )

    y_pred_train = model.predict(X_train, verbose=0)
    y_pred_val = model.predict(X_val, verbose=0)

    row = {
        "trial_id": trial_id,
        "input_window": INPUT_WINDOW,
        "output_window": OUTPUT_WINDOW,
        "MAE_train": mean_absolute_error(y_train, y_pred_train),
        "MAE_val": mean_absolute_error(y_val, y_pred_val),
        "params": model.count_params(),
        "epochs_trained": len(history.history["loss"]),
        **cfg,
    }

    pd.DataFrame(history.history).to_csv(OUTPUT_DIR / f"trial_{trial_id:02d}_history.csv", index=False)
    return row, model


rows = []
best_val = float("inf")
best_trial_id = None
best_config = None

for trial_id in range(1, N_TRIALS + 1):
    cfg = sample_config(trial_id)
    row, model = train_trial(trial_id, cfg)
    rows.append(row)

    partial_df = pd.DataFrame(rows).sort_values("MAE_val")
    partial_df.to_csv(OUTPUT_DIR / "cnn_hyperparameter_trials_partial.csv", index=False)

    if row["MAE_val"] < best_val:
        best_val = row["MAE_val"]
        best_trial_id = trial_id
        best_config = cfg
        model.save(OUTPUT_DIR / "best_cnn_model.keras")
        with open(OUTPUT_DIR / "best_config.json", "w", encoding="utf-8") as f:
            json.dump(best_config, f, indent=2)
        print(f"New best model: trial {trial_id}, MAE_val={best_val:.10f}")

trials_df = pd.DataFrame(rows).sort_values("MAE_val").reset_index(drop=True)
trials_path = OUTPUT_DIR / "cnn_hyperparameter_trials.csv"
trials_df.to_csv(trials_path, index=False)

display(trials_df.head(10))
print("Trials saved to:", trials_path)
print("Best config saved to:", OUTPUT_DIR / "best_config.json")


Trial 1/15
{
  "filters_1": 96,
  "filters_2": 32,
  "filters_3": 64,
  "kernel_1": 5,
  "kernel_2": 3,
  "kernel_3": 3,
  "dilation_2": 1,
  "dilation_3": 2,
  "spatial_dropout": 0.05,
  "dense_1": 256,
  "dense_2": 64,
  "dropout_1": 0.1,
  "dropout_2": 0.05,
  "learning_rate": 0.001,
  "batch_size": 64,
  "trial_id": 1
}
Epoch 1/100


E0000 00:00:1777829822.288623 1765399 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.


  1/205 ━━━━━━━━━━━━━━━━━━━━ 4:05 1s/step - loss: 1.0102 - mae: 1.0102

 10/205 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.6114 - mae: 0.6114 

 20/205 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.4372 - mae: 0.4372

...

 95/103 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0055 - mae: 0.0055

101/103 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0055 - mae: 0.0055

103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0055 - mae: 0.0055 - val_loss: 0.0042 - val_mae: 0.0042 - learning_rate: 3.1250e-05


,trial_id,input_window,output_window,MAE_train,MAE_val,params,epochs_trained,filters_1,filters_2,filters_3,...,kernel_3,dilation_2,dilation_3,spatial_dropout,dense_1,dense_2,dropout_1,dropout_2,learning_rate,batch_size
0,1,30,5,0.005476,0.004151,78327,29,96,32,64,...,3,1,2,0.05,256,64,0.1,0.05,0.0010,64
1,10,30,5,0.005474,0.004151,130647,38,64,64,128,...,5,2,4,0.15,128,128,0.1,0.05,0.0010,256
2,13,30,5,0.005475,0.004151,120279,17,64,64,128,...,3,1,4,0.20,256,32,0.3,0.10,0.0003,128
3,3,30,5,0.005478,0.004152,80951,23,32,32,64,...,3,2,4,0.20,256,128,0.2,0.10,0.0010,128
4,9,30,5,0.005473,0.004152,69559,51,64,32,128,...,3,1,4,0.05,128,64,0.1,0.05,0.0010,128
5,7,30,5,0.005474,0.004152,131927,18,96,64,64,...,3,2,2,0.10,256,128,0.3,0.05,0.0001,64
6,8,30,5,0.005476,0.004153,125751,21,96,32,128,...,5,1,2,0.15,256,64,0.2,0.05,0.0005,256
7,15,30,5,0.005474,0.004153,74231,33,64,96,64,...,3,2,4,0.10,64,128,0.1,0.05,0.0005,128
8,12,30,5,0.005474,0.004153,110167,29,64,96,96,...,3,2,2,0.15,256,32,0.1,0.05,0.0003,64
9,11,30,5,0.005478,0.004153,53079,23,32,32,96,...,5,1,2,0.20,128,32,0.1,0.05,0.0005,64


Trials saved to: /home/hugo/Desktop/Neural-Networks-Forecasting/model/CNN/outputs/cnn_hyperparameter_search_30_5/cnn_hyperparameter_trials.csv
Best config saved to: /home/hugo/Desktop/Neural-Networks-Forecasting/model/CNN/outputs/cnn_hyperparameter_search_30_5/best_config.json


## Final test evaluation of the selected model

The selected model is loaded from disk and evaluated on the untouched test set. This gives the final result of the hyperparameter search.

In [6]:
best_model = keras.models.load_model(OUTPUT_DIR / "best_cnn_model.keras")

y_pred_train = best_model.predict(X_train, verbose=0)
y_pred_val = best_model.predict(X_val, verbose=0)
y_pred_test = best_model.predict(X_test, verbose=0)

best_result = {
    "model": "CNN_Optimized_RandomSearch",
    "selected_trial_id": best_trial_id,
    "input_window": INPUT_WINDOW,
    "output_window": OUTPUT_WINDOW,
    "MAE_train": mean_absolute_error(y_train, y_pred_train),
    "MAE_val": mean_absolute_error(y_val, y_pred_val),
    "MAE_test": mean_absolute_error(y_test, y_pred_test),
    "params": best_model.count_params(),
    "selection_metric": "MAE_val",
}

best_result_df = pd.DataFrame([best_result])
best_result_path = OUTPUT_DIR / "best_cnn_optimized_result.csv"
best_result_df.to_csv(best_result_path, index=False)

comparison_df = None
lr_path = PROJECT_ROOT / "data" / "lr_benchmark.csv"
if lr_path.exists():
    lr = pd.read_csv(lr_path)
    lr_match = lr[(lr["input_window"] == INPUT_WINDOW) & (lr["output_window"] == OUTPUT_WINDOW)]
    if len(lr_match) == 1:
        lr_row = lr_match.iloc[0]
        comparison_df = pd.DataFrame([
            {
                "model": "Linear_Regression_Benchmark",
                "input_window": INPUT_WINDOW,
                "output_window": OUTPUT_WINDOW,
                "MAE_train": lr_row["MAE_train"],
                "MAE_val": np.nan,
                "MAE_test": lr_row["MAE_test"],
                "params": np.nan,
            },
            best_result,
        ])
        comparison_df["improvement_abs_vs_lr"] = comparison_df["MAE_test"].iloc[0] - comparison_df["MAE_test"]
        comparison_df["improvement_pct_vs_lr"] = (
            comparison_df["improvement_abs_vs_lr"] / comparison_df["MAE_test"].iloc[0] * 100
        )
        comparison_path = OUTPUT_DIR / "best_cnn_optimized_vs_lr.csv"
        comparison_df.to_csv(comparison_path, index=False)
        print("Comparison saved to:", comparison_path)

print("Best result saved to:", best_result_path)
display(best_result_df)
if comparison_df is not None:
    display(comparison_df)

Comparison saved to: /home/hugo/Desktop/Neural-Networks-Forecasting/model/CNN/outputs/cnn_hyperparameter_search_30_5/best_cnn_optimized_vs_lr.csv
Best result saved to: /home/hugo/Desktop/Neural-Networks-Forecasting/model/CNN/outputs/cnn_hyperparameter_search_30_5/best_cnn_optimized_result.csv


,model,selected_trial_id,input_window,output_window,MAE_train,MAE_val,MAE_test,params,selection_metric
0,CNN_Optimized_RandomSearch,1,30,5,0.005476,0.004151,0.005578,78327,MAE_val


,model,input_window,output_window,MAE_train,MAE_val,MAE_test,params,selected_trial_id,selection_metric,improvement_abs_vs_lr,improvement_pct_vs_lr
0,Linear_Regression_Benchmark,30,5,0.005337,NaN,0.005877,NaN,NaN,NaN,0.000000,0.000000
1,CNN_Optimized_RandomSearch,30,5,0.005476,0.004151,0.005578,78327.0,1.0,MAE_val,0.000299,5.083225


## How to report this

This notebook supports the following statement in the report:

> A controlled random search was performed for the CNN model. The tested hyperparameters included the number of convolutional filters, kernel sizes, dilation rates, dropout levels, dense layer widths, learning rate and batch size. The model was selected using validation MAE, while the test set was reserved for final evaluation only.